# EasyRAG Retrieval Metrics

## What you will do

- build a tiny deterministic retrieval eval set
- run `evaluate_retrieval()` over that set
- compare the aggregate report with the underlying metric helpers

## Why this matters

Optimization should start from measurement. This notebook keeps retrieval evaluation reproducible by using a tiny corpus and explicit expected document IDs.


## Source code anchors

- `easyrag.rag.types.EvalCase`
- `easyrag.rag.evaluation.runner.evaluate_retrieval`
- `easyrag.rag.evaluation.metrics.hit_rate_at_k`
- `easyrag.rag.evaluation.metrics.recall_at_k`
- `easyrag.rag.evaluation.metrics.precision_at_k`
- `easyrag.rag.evaluation.metrics.mrr_at_k`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.evaluation import evaluate_retrieval
from easyrag.rag.evaluation.metrics import hit_rate_at_k, mrr_at_k, precision_at_k, recall_at_k


## Deterministic path

The eval set is intentionally small. That makes the per-case reports readable enough that you can connect the metric values to real retrieved citations.


In [ ]:
eval_tmp = tempfile.TemporaryDirectory()
eval_rag = EasyRAG(
    working_dir=eval_tmp.name,
    workspace="retrieval-eval-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(eval_rag.initialize_storages())
run_sync(
    eval_rag.ainsert(
        [
            "# Architecture\nEasyRAG uses query rewrite for grounded retrieval.\n",
            "# Storage\nRetrieved evidence is packed before answer synthesis.\n",
            "# Policy\nCitation-aware outputs preserve traceability.\n",
        ],
        ids=["doc::architecture", "doc::storage", "doc::policy"],
        file_paths=["docs/architecture.md", "docs/storage.md", "docs/policy.md"],
    )
)

cases = [
    EvalCase(question="What uses query rewrite?", expected_document_ids=("doc::architecture",)),
    EvalCase(question="What is packed before answer synthesis?", expected_document_ids=("doc::storage",)),
    EvalCase(question="Which policy keeps outputs traceable?", expected_document_ids=("doc::policy",)),
]
retrieval_report = run_sync(
    evaluate_retrieval(
        eval_rag,
        cases,
        QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3),
    )
)

print("Aggregate metrics")
_print_json(retrieval_report["metrics"])
print("\nPer-case reports")
_print_json(retrieval_report["cases"])


## Output inspection

The helper metrics below recompute the first case from its boolean relevance list. This is useful when you want to sanity-check the aggregate report on a single example.


In [ ]:
first_case = retrieval_report["cases"][0]
first_relevances = [document_id in first_case["expected_document_ids"] for document_id in first_case["retrieved_document_ids"]]
manual_metrics = {
    "hit_rate": hit_rate_at_k(first_relevances),
    "recall_at_k": recall_at_k(first_relevances, expected_total=len(first_case["expected_document_ids"])),
    "precision_at_k": precision_at_k(first_relevances),
    "mrr_at_k": mrr_at_k(first_relevances),
}
_print_json({"relevances": first_relevances, "manual_metrics": manual_metrics})


## Provider-backed path

If provider-backed defaults are available, the same evaluation runner can execute over a provider-backed retriever. The cell is still small on purpose: one corpus, one case set, one metrics report.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(working_dir=provider_tmp.name, workspace="provider-retrieval-eval-demo")
    try:
        run_sync(provider_rag.initialize_storages())
        run_sync(
            provider_rag.ainsert(
                [
                    "# Retrieval\nGrounded retrieval uses rewrite and chunk ranking.\n",
                    "# Storage\nPacked evidence stays available for the answer layer.\n",
                ],
                ids=["doc::retrieval", "doc::storage"],
                file_paths=["docs/retrieval.md", "docs/storage.md"],
            )
        )
        provider_report = run_sync(
            evaluate_retrieval(
                provider_rag,
                [EvalCase(question="What uses rewrite?", expected_document_ids=("doc::retrieval",))],
                QueryParam(mode="hybrid", chunk_top_k=2),
            )
        )
        _print_json(provider_report["metrics"])
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()


## What changed and why

Once the eval cases are explicit, the report becomes actionable. `hit_rate` tells you whether anything relevant surfaced at all. `recall_at_k` tells you how much of the expected evidence you found. `precision_at_k` tells you how noisy the ranked prefix is. `mrr_at_k` tells you how early the first relevant hit appears.


In [ ]:
run_sync(eval_rag.finalize_storages())
eval_tmp.cleanup()
print("Cleaned up the deterministic retrieval-eval workspace.")


## Next steps

- Continue with [06_04_answer_grounding_and_faithfulness.ipynb](06_04_answer_grounding_and_faithfulness.ipynb) to separate retrieval quality from answer quality.
- Read [06-evaluation-overview.md](../../docs/06-evaluation-overview.md) for the chapter-level framing of retrieval, answer, latency, and regression evaluation.
- Revisit [04_05_fusion_rerank_and_topk.ipynb](../04_retrieval/04_05_fusion_rerank_and_topk.ipynb) if you want to trace a ranking regression back to fusion or rerank behavior.
